In [ ]:
import wandb
import os
import sys
import torch
import wandb
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sys.path.append(os.path.abspath(os.path.join("..")))
from src.LSTM.preprocess import (
    load_data,
    encode_features,
    scale_data_new,
    create_sequences,
)
from src.LSTM.network_variants import LSTMAttentionModel

def log_run_data(
    run_id, entity="ml-for-data-analytics-project", project="energy-forecasting"
):
    """Downloads model, scaler, and prediction table from W&B."""
    api = wandb.Api()
    run = api.run(f"{entity}/{project}/{run_id}")
    config = run.config

    artifact = api.artifact(
        f"{entity}/{project}/trained_model_{config["run_name"]}:latest"
    )
    artifact_dir = artifact.download()

    table_artifact = [a for a in run.logged_artifacts() if "predictions" in a.name][0]
    df_logged = table_artifact.get("predictions").get_dataframe()

    scaler = joblib.load(os.path.join(artifact_dir, "scaler.joblib"))
    state_dict = torch.load(os.path.join(artifact_dir, "model.pth"), map_location="cpu")

    parent_dir = os.path.dirname(os.getcwd())
    train_df = load_data(os.path.join(parent_dir, config["data"]["train_path"]))
    test_df = load_data(os.path.join(parent_dir, config["data"]["test_path"]))

    train_df, test_df = encode_features(
        train_df, test_df, resolution=config["data"]["resolution"]
    )
    train_df, test_df, _ = scale_data_new(train_df, test_df)

    lags = config["data"]["lags"]
    _, _, _, Xp_test, Xf_test, _ = create_sequences(
        train_df, test_df, k=lags[-1], resolution=config["data"]["resolution"]
    )

    return (
        run,
        config,
        Xp_test,
        Xf_test,
        df_logged,
        scaler,
        train_df.columns,
        state_dict,
    )